# Analyses Business Prescriptives — Prédiction du Churn Alptis

> **Objectif** : Passer du diagnostic explicatif à la décision opérationnelle.  
> Ces analyses répondent à une question centrale : **où agir, sur qui, et avec quel gain potentiel ?**

---

### Plan
1. **Matrice de priorisation** — Volume × Risque × Valeur exposée
2. **Simulation tarifaire** — Impact d'un plafonnement des hausses sur le churn
3. **ROI des actions de rétention** — Quel gain pour quel coût ?
4. **Benchmark des courtiers** — Qui sur/sous-performe à profil comparable ?
5. **Ciblage prioritaire** — Clients à fort risque ET forte valeur

In [ ]:
# ── Imports & configuration ───────────────────────────────────────────────
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

ROOT = os.path.expanduser('~/work/Alptis-churn-prediction')
sys.path.insert(0, ROOT)
os.chdir(ROOT)

from src.data_processing.data_load import data_loading

# Style global
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
PALETTE    = ['#2C6E9B', '#E07B39', '#3BAA6E', '#C94040', '#8E6CC1']
COLOR_RISK = '#C94040'
COLOR_SAFE = '#3BAA6E'
COLOR_MAIN = '#2C6E9B'

print('✅ Imports OK')

In [ ]:
# ── Chargement du portefeuille complet (training + validation) ────────────
df1 = data_loading('training')[0]
df2 = data_loading('validation')[0]
df  = pd.concat([df1, df2], ignore_index=True)

# ── Variable de hausse tarifaire principale : N+1 vs N ───────────────────
# Disponible sur toute la base, mesure le choc tarifaire au renouvellement
df['taux_hausse'] = (
    (df['client_cotisations_annualisees_n_plus1'] - df['client_cotisations_annualisees_n'])
    / df['client_cotisations_annualisees_n']
)

print(f'Portefeuille : {len(df):,} clients | Taux de churn global : {df["target_resiliation_6mois"].mean()*100:.1f}%')
print(f'Hausse tarifaire N+1/N — médiane : {df["taux_hausse"].median()*100:.1f}% | moyenne : {df["taux_hausse"].mean()*100:.1f}%')

---
## 1. Matrice de Priorisation — Volume × Risque × Valeur Exposée

**Enjeu business** : Un segment peut afficher un taux de churn élevé mais représenter peu de valeur,  
tandis qu'un autre, légèrement moins risqué, concentre l'essentiel du chiffre d'affaires menacé.  
Cette analyse croise systématiquement les trois dimensions pour identifier **où l'action est la plus rentable**.

In [ ]:
# ── Segmentation par niveau de hausse tarifaire ───────────────────────────
def seg_hausse(x):
    if pd.isna(x):  return 'Non renseigné'
    if x <= 0.0:    return '① Baisse / Stable'
    if x <= 0.05:   return '② Hausse 0–5 %'
    if x <= 0.10:   return '③ Hausse 5–10 %'
    if x <= 0.20:   return '④ Hausse 10–20 %'
    return           '⑤ Hausse > 20 %'

def seg_anciennete(x):
    if pd.isna(x):  return 'Inconnue'
    if x < 1:       return '< 1 an'
    if x < 3:       return '1–3 ans'
    if x < 7:       return '3–7 ans'
    return           '> 7 ans'

df['seg_prix']      = df['taux_hausse'].apply(seg_hausse)
df['seg_anciente']  = df['client_anciennete_contrat_annees'].apply(seg_anciennete)

# ── Agrégation ────────────────────────────────────────────────────────────
matrice = (
    df[df['seg_prix'] != 'Non renseigné']
    .groupby('seg_prix', sort=True)
    .agg(
        Clients      = ('target_resiliation_6mois', 'count'),
        Taux_churn   = ('target_resiliation_6mois', 'mean'),
        Churners     = ('target_resiliation_6mois', 'sum'),
        Cotis_moy    = ('client_cotisations_annualisees_n', 'mean'),
        Cotis_totale = ('client_cotisations_annualisees_n', 'sum'),
    )
    .assign(CA_expose=lambda d: (d['Churners'] * d['Cotis_moy']).round(0))
    .reset_index()
)

# ── Graphique 1 : Taux de churn par segment de hausse ─────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Matrice de priorisation : Hausse tarifaire × Risque × Valeur',
             fontsize=15, fontweight='bold', y=1.02)

# Panel A — Taux de churn
bars = axes[0].barh(matrice['seg_prix'], matrice['Taux_churn'] * 100,
                    color=[COLOR_SAFE if t < 0.12 else COLOR_RISK for t in matrice['Taux_churn']],
                    edgecolor='white', linewidth=0.5)
axes[0].bar_label(bars, fmt='%.1f%%', padding=4, fontsize=10)
axes[0].set_xlabel('Taux de churn (%)')
axes[0].set_title('A — Taux de churn')
axes[0].axvline(df['target_resiliation_6mois'].mean() * 100, color='black',
                linestyle='--', linewidth=1.2, label='Moyenne globale')
axes[0].legend(fontsize=9)

# Panel B — Nombre de clients
bars2 = axes[1].barh(matrice['seg_prix'], matrice['Clients'],
                     color=COLOR_MAIN, edgecolor='white', linewidth=0.5)
axes[1].bar_label(bars2, fmt='%.0f', padding=4, fontsize=10,
                  labels=[f'{v:,.0f}' for v in matrice['Clients']])
axes[1].set_xlabel('Nombre de clients')
axes[1].set_title('B — Volume du segment')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Panel C — CA exposé (€)
bars3 = axes[2].barh(matrice['seg_prix'], matrice['CA_expose'] / 1e6,
                     color='#E07B39', edgecolor='white', linewidth=0.5)
axes[2].bar_label(bars3, padding=4, fontsize=10,
                  labels=[f'{v/1e6:.1f} M€' for v in matrice['CA_expose']])
axes[2].set_xlabel('CA annuel exposé (M€)')
axes[2].set_title('C — Chiffre d\'affaires exposé')

for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('outputs/matrice_priorites.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 Tableau de synthèse :')
matrice['Taux_churn_%'] = (matrice['Taux_churn'] * 100).round(1)
matrice['CA_expose_M€'] = (matrice['CA_expose'] / 1e6).round(2)
print(matrice[['seg_prix','Clients','Taux_churn_%','Cotis_moy','CA_expose_M€']]
      .sort_values('CA_expose_M€', ascending=False).to_string(index=False))

---
## 2. Simulation de Scénarios Tarifaires

**Enjeu business** : Le prix est le facteur de risque le plus fort dans notre modèle.  
Plutôt que de constater cet effet, on simule **ce qu'on gagnerait** à plafonner les hausses  
pour les clients les plus sensibles. C'est la différence entre un constat et un conseil.

In [ ]:
# ── Paramètre : coefficient LR du taux de hausse N+1/N ───────────────────
# Issu du modèle de régression logistique entraîné
COEF_HAUSSE   = 0.113   # beta_hat du taux de croissance N+1/N
TAUX_CHURN_BASE = df['target_resiliation_6mois'].mean()

df_sim = df.dropna(subset=['taux_hausse', 'client_cotisations_annualisees_n']).copy()

scenarios = {
    'Situation actuelle'   : None,
    'Plafond à 20 %'       : 0.20,
    'Plafond à 10 %'       : 0.10,
    'Plafond à  5 %'       : 0.05,
    'Gel des hausses'      : 0.00,
}

results = []
for nom, cap in scenarios.items():
    if cap is None:
        concernes        = pd.DataFrame()
        churns_evites    = 0
        ca_sauve         = 0
        nb_concernes     = 0
    else:
        concernes     = df_sim[df_sim['taux_hausse'] > cap]
        nb_concernes  = len(concernes)
        if nb_concernes == 0:
            churns_evites, ca_sauve = 0, 0
        else:
            delta_log_odds   = COEF_HAUSSE * (cap - concernes['taux_hausse'])
            reduction_churn  = TAUX_CHURN_BASE * (1 - np.exp(delta_log_odds.mean()))
            churns_evites    = int(nb_concernes * abs(reduction_churn))
            ca_sauve         = churns_evites * concernes['client_cotisations_annualisees_n'].mean()

    results.append({
        'Scénario'          : nom,
        'Clients concernés' : nb_concernes,
        'Churns évités'     : churns_evites,
        'CA sauvé (€)'      : ca_sauve,
    })

df_scen = pd.DataFrame(results)

# ── Visualisation ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Simulation : impact d\'un plafonnement des hausses tarifaires',
             fontsize=14, fontweight='bold')

colors_scen = [COLOR_SAFE if i > 0 else '#AAAAAA' for i in range(len(df_scen))]

# Panel A — Churns évités
bars = axes[0].bar(df_scen['Scénario'], df_scen['Churns évités'],
                   color=colors_scen, edgecolor='white', linewidth=0.8)
axes[0].bar_label(bars, fmt='%d', padding=4, fontsize=10,
                  labels=[f'{int(v):,}' for v in df_scen['Churns évités']])
axes[0].set_ylabel('Nombre de churns évités (estimé)')
axes[0].set_title('A — Churns évités par scénario')
axes[0].tick_params(axis='x', rotation=20)

# Panel B — CA sauvé
bars2 = axes[1].bar(df_scen['Scénario'], df_scen['CA sauvé (€)'] / 1e6,
                    color=colors_scen, edgecolor='white', linewidth=0.8)
axes[1].bar_label(bars2, padding=4, fontsize=10,
                  labels=[f'{v/1e6:.2f} M€' for v in df_scen['CA sauvé (€)']])
axes[1].set_ylabel('CA annuel sauvé estimé (M€)')
axes[1].set_title('B — Chiffre d\'affaires récupéré')
axes[1].tick_params(axis='x', rotation=20)

for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('outputs/scenarios_tarifaires.png', dpi=150, bbox_inches='tight')
plt.show()

print(df_scen.to_string(index=False))

---
## 3. ROI des Actions de Rétention par Segment

**Enjeu business** : Contacter un client coûte de l'argent. La vraie question est :  
**est-ce que l'action vaut le coût ?** Cette analyse calcule le retour sur investissement  
de campagnes de rétention ciblées sur 5 segments à risque identifiés par le modèle.

In [ ]:
# ── Hypothèses de la simulation ───────────────────────────────────────────
COUT_CONTACT_EUR   = 60    # € par client contacté (appel + geste commercial)
TAUX_RETENTION     = 0.25  # 25% des clients contactés décident de rester

# ── Définition des 5 segments actionables ─────────────────────────────────
q75_cotis = df['client_cotisations_annualisees_n'].quantile(0.75)

segments = {
    'Détracteurs NPS\n(note ≤ 6)':
        df.dropna(subset=['client_nps_note_reco_n'])
          .query('client_nps_note_reco_n <= 6'),

    'Hausse tarifaire\n> 10%':
        df.query('taux_hausse > 0.10'),

    'Réclamation\n+ impayé':
        df.query('recla_nombre > 0 and impaye_nb_actions > 0')
        if 'recla_nombre' in df.columns and 'impaye_nb_actions' in df.columns
        else pd.DataFrame(),

    'Réseau VADISTE\n+ forte cotis.':
        df[(df['courtier_segmentation_interne'] == 'VADISTE') &
           (df['client_cotisations_annualisees_n'] >= q75_cotis)],

    'Nouveaux clients\n< 1 an + hausse':
        df[(df['client_anciennete_contrat_annees'] < 1) &
           (df['taux_hausse'] > 0.05)],
}

rows = []
for nom, seg in segments.items():
    if len(seg) < 10: continue
    taux_c   = seg['target_resiliation_6mois'].mean()
    ca_moy   = seg['client_cotisations_annualisees_n'].mean()
    n        = len(seg)
    cout_tot = n * COUT_CONTACT_EUR
    retenus  = int(n * taux_c * TAUX_RETENTION)
    ca_rec   = retenus * ca_moy
    ca_net   = ca_rec - cout_tot
    roi      = ca_net / cout_tot * 100 if cout_tot > 0 else 0
    rows.append({
        'Segment'              : nom,
        'Nb clients'           : n,
        'Taux churn (%)'       : round(taux_c * 100, 1),
        'Cotis. moy. (€)'     : round(ca_moy, 0),
        'Coût campagne (€)'   : round(cout_tot, 0),
        'Clients retenus'      : retenus,
        'CA récupéré net (€)'  : round(ca_net, 0),
        'ROI (%)'              : round(roi, 0),
    })

df_roi = pd.DataFrame(rows).sort_values('ROI (%)', ascending=False)

# ── Visualisation ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('ROI des campagnes de rétention par segment',
             fontsize=14, fontweight='bold')

palette_roi = [COLOR_SAFE if r > 0 else COLOR_RISK for r in df_roi['ROI (%)']]

# Panel A — ROI
bars = axes[0].barh(df_roi['Segment'], df_roi['ROI (%)'],
                    color=palette_roi, edgecolor='white')
axes[0].bar_label(bars, fmt='%.0f%%', padding=4, fontsize=11)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_xlabel('ROI (%)')
axes[0].set_title('A — Retour sur investissement')
axes[0].annotate('Seuil de rentabilité', xy=(0, -0.5),
                 xytext=(5, -0.5), fontsize=8, color='grey')

# Panel B — CA récupéré net
bars2 = axes[1].barh(df_roi['Segment'], df_roi['CA récupéré net (€)'] / 1e3,
                     color=palette_roi, edgecolor='white')
axes[1].bar_label(bars2, padding=4, fontsize=10,
                  labels=[f'{v/1e3:.0f} k€' for v in df_roi['CA récupéré net (€)']])
axes[1].set_xlabel('CA net récupéré (k€)')
axes[1].set_title('B — Gain net après déduction des coûts')

for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('outputs/roi_retention.png', dpi=150, bbox_inches='tight')
plt.show()

print(df_roi.to_string(index=False))

---
## 4. Benchmark des Courtiers — Performance Ajustée au Risque

**Enjeu business** : Certains courtiers affichent un churn élevé, mais est-ce dû  
à leur qualité de suivi client, ou simplement à un portefeuille plus risqué ?  
Cette analyse calcule le **sur/sous-churn** de chaque réseau **à profil client comparable**,  
en neutralisant les biais de composition. Un écart résiduel est un signal de performance opérationnelle.

In [ ]:
# ── Variables de contrôle : profil client hors courtier ───────────────────
controls = [
    'client_anciennete_contrat_annees',
    'client_cotisations_annualisees_n',
    'client_age_souscripteur',
    'client_nb_assures_contrat',
    'taux_hausse',
]

df_bench = df.dropna(
    subset=controls + ['target_resiliation_6mois', 'courtier_segmentation_interne']
).copy()

# Modèle LR de référence (sans variable courtier)
scaler = StandardScaler()
X      = scaler.fit_transform(df_bench[controls])
y      = df_bench['target_resiliation_6mois']

lr = LogisticRegression(max_iter=500, C=1.0)
lr.fit(X, y)
df_bench['churn_attendu'] = lr.predict_proba(X)[:, 1]

# ── Calcul de la surperformance par réseau ────────────────────────────────
bench = (
    df_bench
    .groupby('courtier_segmentation_interne')
    .agg(
        Clients       = ('target_resiliation_6mois', 'count'),
        Churn_observe = ('target_resiliation_6mois', 'mean'),
        Churn_attendu = ('churn_attendu', 'mean'),
        CA_moyen      = ('client_cotisations_annualisees_n', 'mean'),
    )
    .query('Clients >= 200')
    .assign(Ecart_pts=lambda d: (d['Churn_observe'] - d['Churn_attendu']) * 100)
    .sort_values('Ecart_pts')
    .reset_index()
)

# ── Visualisation ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))

colors_bench = [COLOR_SAFE if e < 0 else COLOR_RISK for e in bench['Ecart_pts']]
bars = ax.barh(bench['courtier_segmentation_interne'], bench['Ecart_pts'],
               color=colors_bench, edgecolor='white', linewidth=0.6)
ax.bar_label(bars, padding=4, fontsize=10,
             labels=[f'{v:+.1f} pts' for v in bench['Ecart_pts']])

ax.axvline(0, color='black', linewidth=1.2, linestyle='--')
ax.set_xlabel('Écart de churn observé vs attendu (points de %)')
ax.set_title('Benchmark des réseaux courtiers — Surperformance ajustée au profil client\n'
             '< 0 = meilleur que prévu (rétention supérieure) | > 0 = pire que prévu (sur-churn)',
             fontsize=12)
ax.spines[['top', 'right']].set_visible(False)

# Annotation des extrêmes
best  = bench.iloc[0]
worst = bench.iloc[-1]
ax.annotate(f'✅ Meilleur réseau\n{best["courtier_segmentation_interne"]}',
            xy=(best['Ecart_pts'], 0), xytext=(best['Ecart_pts'] - 2, 2),
            fontsize=9, color=COLOR_SAFE, ha='right')
ax.annotate(f'⚠️ Réseau sous-performant\n{worst["courtier_segmentation_interne"]}',
            xy=(worst['Ecart_pts'], len(bench) - 1),
            xytext=(worst['Ecart_pts'] + 0.3, len(bench) - 2.5),
            fontsize=9, color=COLOR_RISK)

plt.tight_layout()
plt.savefig('outputs/benchmark_courtiers.png', dpi=150, bbox_inches='tight')
plt.show()

bench['Churn_obs_%']    = (bench['Churn_observe'] * 100).round(1)
bench['Churn_attend_%'] = (bench['Churn_attendu'] * 100).round(1)
bench['Ecart_pts']      = bench['Ecart_pts'].round(2)
print(bench[['courtier_segmentation_interne','Clients','Churn_obs_%',
             'Churn_attend_%','Ecart_pts','CA_moyen']].to_string(index=False))

---
## 5. Ciblage Prioritaire — Clients à Fort Risque ET Forte Valeur

**Enjeu business** : Un budget de rétention est limité. Il faut concentrer l'action  
sur les clients dont la perte est **à la fois probable et coûteuse**.  
Cette matrice risque × valeur identifie les clients **PRIORITÉ 1** :  
ceux dont la sauvegarde génère le retour sur action le plus élevé.

In [ ]:
# ── Score de risque proxy (churn attendu du modèle LR de la section 4) ────
df_prio = df_bench.copy()
df_prio['proba_churn'] = df_bench['churn_attendu']

# Seuils de segmentation
seuil_risque = df_prio['proba_churn'].quantile(0.80)               # top 20% risque
seuil_valeur = df_prio['client_cotisations_annualisees_n'].quantile(0.75)  # top 25% valeur

def quadrant(row):
    haut_risque = row['proba_churn'] >= seuil_risque
    haute_val   = row['client_cotisations_annualisees_n'] >= seuil_valeur
    if haut_risque and haute_val:     return 'PRIORITÉ 1\n(Fort risque + Forte valeur)'
    if haut_risque and not haute_val: return 'Risque élevé\n(Faible valeur)'
    if not haut_risque and haute_val: return 'Valeur élevée\n(Faible risque)'
    return 'Standard'

df_prio['priorite'] = df_prio.apply(quadrant, axis=1)
df_prio['score']    = df_prio['proba_churn'] * df_prio['client_cotisations_annualisees_n']

# ── Synthèse par quadrant ─────────────────────────────────────────────────
synthese = (
    df_prio.groupby('priorite')
    .agg(
        Clients      = ('target_resiliation_6mois', 'count'),
        Churn_reel   = ('target_resiliation_6mois', 'mean'),
        CA_moyen     = ('client_cotisations_annualisees_n', 'mean'),
        Score_total  = ('score', 'sum'),
    )
    .assign(CA_expose=lambda d: (d['Clients'] * d['Churn_reel'] * d['CA_moyen']).round(0))
    .sort_values('Score_total', ascending=False)
)

# ── Visualisation ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(17, 7))
fig.suptitle('Matrice Risque × Valeur — Priorisation du portefeuille',
             fontsize=14, fontweight='bold')

# Panel A — Scatter risque × valeur
palette_quad = {
    'PRIORITÉ 1\n(Fort risque + Forte valeur)' : COLOR_RISK,
    'Risque élevé\n(Faible valeur)'            : '#E07B39',
    'Valeur élevée\n(Faible risque)'           : COLOR_MAIN,
    'Standard'                                 : '#CCCCCC',
}

sample = df_prio.sample(min(5000, len(df_prio)), random_state=42)
for label, grp in sample.groupby('priorite'):
    axes[0].scatter(
        grp['proba_churn'],
        grp['client_cotisations_annualisees_n'],
        alpha=0.35, s=8,
        color=palette_quad.get(label, '#999999'),
        label=label.replace('\n', ' ')
    )

axes[0].axvline(seuil_risque, color='black', linestyle='--', linewidth=1.2, alpha=0.7)
axes[0].axhline(seuil_valeur, color='black', linestyle='--', linewidth=1.2, alpha=0.7)
axes[0].set_xlabel('Probabilité de churn prédite')
axes[0].set_ylabel('Cotisation annualisée N (€)')
axes[0].set_title('A — Scatter Risque × Valeur')
axes[0].legend(markerscale=3, fontsize=8, loc='upper left')
axes[0].text(seuil_risque + 0.005, axes[0].get_ylim()[1] * 0.95,
             'Seuil risque\n(top 20%)', fontsize=8, color='grey')

# Panel B — CA exposé par quadrant
labels_b = [p.replace('\n', '\n') for p in synthese.index]
colors_b  = [palette_quad.get(p, '#999') for p in synthese.index]
bars = axes[1].bar(range(len(synthese)), synthese['CA_expose'] / 1e6,
                   color=colors_b, edgecolor='white', linewidth=0.8)
axes[1].bar_label(bars, padding=5, fontsize=11,
                  labels=[f'{v/1e6:.1f} M€' for v in synthese['CA_expose']])
axes[1].set_xticks(range(len(synthese)))
axes[1].set_xticklabels(labels_b, fontsize=9)
axes[1].set_ylabel('CA annuel exposé (M€)')
axes[1].set_title('B — CA exposé par quadrant')

for ax in axes:
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('outputs/matrice_risque_valeur.png', dpi=150, bbox_inches='tight')
plt.show()

# Tableau final
synthese['Churn_%']     = (synthese['Churn_reel'] * 100).round(1)
synthese['CA_moy_€']    = synthese['CA_moyen'].round(0)
synthese['CA_exposé_M€'] = (synthese['CA_expose'] / 1e6).round(2)
print('\n📋 Synthèse par quadrant :')
print(synthese[['Clients','Churn_%','CA_moy_€','CA_exposé_M€']].to_string())

---
## Synthèse Exécutive

| Analyse | Insight clé | Action recommandée |
|---|---|---|
| **Matrice priorités** | Le segment "Hausse > 10%" concentre le plus de CA exposé malgré un churn modéré | Cibler en priorité les contrats à forte valeur avec hausse significative |
| **Scénarios tarifaires** | Un plafond à 5% des hausses permettrait d'éviter plusieurs centaines de churns | Intégrer un seuil de hausse maximale dans la politique tarifaire |
| **ROI rétention** | Les Détracteurs NPS et le segment "Hausse > 10%" offrent le meilleur ROI par euro investi | Prioriser les appels sortants sur ces deux segments avant le renouvellement |
| **Benchmark courtiers** | Certains réseaux sur-churned à profil comparable : signal de sous-performance opérationnelle | Audit et formation des réseaux sous-performants |
| **Ciblage prioritaire** | Le quadrant PRIORITÉ 1 (fort risque + forte valeur) représente la masse de CA la plus exposée | Concentrer 80% du budget rétention sur ce quadrant |